Calculating the per-patient differences across timepoints to determine whether there are significant changes in symptoms for individual patients over time.
---

In [ ]:
import pandas as pd

file_path = "df_symptoms.xlsx"
sheet_name = "Sheet1"

id_col = "PatientID"
timepoints_col = "Timepoint"

tp1 = "Baseline"
tp2 = "End Part 1"
tp3 = "End Part 2"
tp4 = "End Part 3"

out_path = "Per_patient_differences_PGI_PTHS.xlsx"
category = "Group"

In [ ]:
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Keep only available timepoints of interest
wanted_tps = [tp1, tp2, tp3, tp4]
wanted_tps = [t for t in wanted_tps if t in df[timepoints_col].unique()]
df = df[df[timepoints_col].isin(wanted_tps)].copy()

# first 4 columns are metadata; remaining are features
feature_cols = df.columns[4:]

In [ ]:
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

# Convert to wide format: one row per patient
wide = (
    df.groupby([id_col, timepoints_col], as_index=True)[feature_cols]
      .mean()
      .unstack(timepoints_col)
)

def compute_diff(wide_df, from_tp, to_tp):

    # Return empty dataframe if either timepoint is missing
    if from_tp not in wide_df.columns.get_level_values(1) or to_tp not in wide_df.columns.get_level_values(1):
        return pd.DataFrame(columns=[id_col, "tp_from", "tp_to"] + feature_cols)

    to_vals = wide_df.xs(to_tp, axis=1, level=timepoints_col)
    from_vals = wide_df.xs(from_tp, axis=1, level=timepoints_col)

    # Compute per-patient differences
    diff_df = to_vals.reindex(columns=feature_cols) - from_vals.reindex(columns=feature_cols)

    diff_df = diff_df.reset_index()
    diff_df.insert(1, "tp_from", from_tp)
    diff_df.insert(2, "tp_to", to_tp)

    return diff_df

In [ ]:
diff_p1 = compute_diff(wide, tp2, tp1)
diff_p2 = compute_diff(wide, tp3, tp1)
diff_p3 = compute_diff(wide, tp4, tp1)

out = pd.concat([diff_p1, diff_p2, diff_p3], ignore_index=True)

# Add group/category information back to output
mtt = df.groupby(id_col)[category].first().reset_index()
out = out.merge(mtt, on=id_col, how="left")

ordered_cols = [id_col, "tp_from", "tp_to"] + [category] + feature_cols
out = out[ordered_cols]

out.to_excel(out_path, index=False)

Calculating p-value using independent t-test with Unequal variance
------

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_excel("Per_patient_differences.xlsx", sheet_name="Sheet1")

# Symptom columns to test
symptom_cols = df.columns[4:13].tolist()
timepoints = ["End Part 1", "End Part 2", "End Part 3"]

results = []

for tp in timepoints:
    for feature in symptom_cols:
        g_placebo = df.loc[(df["Group"] == "Placebo") & (df["tp_from"] == tp), feature].dropna()         # Split data into placebo and treated groups
        g_treated = df.loc[(df["Group"] == "Treated") & (df["tp_from"] == tp), feature].dropna()

        n1, n2 =len(g_treated), len(g_placebo)

        if n1 == 0 or n2 == 0:         # Skip if either group has no data
            results.append([feature, tp, np.nan, np.nan, np.nan, np.nan])
            continue

        t_stat, p_value = stats.ttest_ind(g_treated, g_placebo, equal_var=False, alternative="two-sided")         # Welch's t-test (unequal variance)
        mean1, mean2 = g_treated.mean(), g_placebo.mean()
        s1, s2 = g_treated.std(ddof=1), g_placebo.std(ddof=1)

        pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))         # Compute Cohen's d effect size
        cohens_d = (mean1 - mean2) / pooled_sd if pooled_sd > 0 else np.nan

        results.append([feature, tp, p_value, t_stat, cohens_d,abs(cohens_d)])

results_df = pd.DataFrame(
    results,
    columns=["feature", "timepoint", "p_value", "t_statistic", "cohens_d","abs_cohens_d"]
)

results_df.to_excel("independent_ttest_with_unequal_var.xlsx", index=False)
print(results_df)

Here we are assuming unequal variance but also using pooled standard deviation to report the effect size. When interpreting the results, use the effect size with caution.